In [1]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv
from prefect import flow

load_dotenv()

ROOT_DIR = Path.cwd().parent
ENV = os.getenv("ENV", default="dev")
os.chdir(ROOT_DIR)

print(f"Current Working Directory: {ROOT_DIR}")

Current Working Directory: c:\Users\nikhi\Documents\Projects\HybridRAG-Bench


# Parse and Chunk

In [ ]:
INPUT_DIR = ROOT_DIR / "data" / ENV / "pdfs"
OUTPUT_DIR = ROOT_DIR / "data" / ENV / "parsed"

pdf_files = sorted(INPUT_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF(s) in {INPUT_DIR}:")
for f in pdf_files:
    print(f"  - {f.name}")

In [ ]:
from src.ingestion_steps import parse_and_chunk


@flow(name="test-parse-and-chunk", log_prints=True)
def test_parse_and_chunk(file_paths: list[Path], output_dir: Path) -> list[Path]:
    """Thin wrapper flow so the Prefect task has a flow context."""
    return parse_and_chunk(file_paths=file_paths, output_dir=output_dir)


output_paths = test_parse_and_chunk(file_paths=pdf_files, output_dir=OUTPUT_DIR)
print(f"\nOutput files ({len(output_paths)}):")
for p in output_paths:
    print(f"  - {p}")

In [ ]:
for output_path in output_paths:
    with open(output_path) as f:
        chunks = json.load(f)

    print(f"\n{'='*60}")
    print(f"File: {output_path.name}  |  Chunks: {len(chunks)}")
    print(f"{'='*60}")

    for i, chunk in enumerate(chunks[:3]):
        text = chunk.get("text", "")
        element_type = chunk.get("type", "unknown")
        # Extract prefix from chunk["text"], i.e., content between "Prefix: " and first "; "
        text = chunk.get("text", "")
        prefix = ""
        prefix_marker = "Prefix: "
        if prefix_marker in text:
            start = text.find(prefix_marker) + len(prefix_marker)
            end = text.find("; ", start)
            if end != -1:
                prefix = text[start:end]
        print(f"\n--- Chunk {i} ({element_type}) ---")
        if prefix:
            print(f"Prefix: {prefix}")
        
        original_text = text
        original_text_marker = "Original: "
        if original_text_marker in text:
            start = text.find(original_text_marker) + len(original_text_marker)
            original_text = text[start:]
        print(f"Original: {original_text}")

    if len(chunks) > 3:
        print(f"\n... and {len(chunks) - 3} more chunk(s)")

# Clean Chunks

In [2]:
PARSED_DIR = ROOT_DIR / "data" / ENV / "parsed"
CLEANED_DIR = ROOT_DIR / "data" / ENV / "cleaned"

parsed_paths = sorted(PARSED_DIR.glob("*.json"))
print(f"Found {len(parsed_paths)} parsed file(s) in {PARSED_DIR}:")
for p in parsed_paths:
    print(f"  - {p.name}")

Found 2 parsed file(s) in c:\Users\nikhi\Documents\Projects\HybridRAG-Bench\data\dev\parsed:
  - Attention-Is-All-You-Need_170603762v7-9ab76e8d.pdf.json
  - BERT_181004805v2-d4016745.pdf.json


In [3]:
from src.ingestion_steps import clean_chunks


@flow(name="test-clean-chunks", log_prints=True)
def test_clean_chunks(parsed_paths: list[Path], output_dir: Path) -> list[Path]:
    """Thin wrapper flow so the Prefect task has a flow context."""
    return clean_chunks(parsed_paths=parsed_paths, output_dir=output_dir)


cleaned_paths = test_clean_chunks(parsed_paths=parsed_paths, output_dir=CLEANED_DIR)
print(f"\nCleaned files ({len(cleaned_paths)}):")
for p in cleaned_paths:
    print(f"  - {p}")

c:\Users\nikhi\Documents\Projects\HybridRAG-Bench\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


20:00:52.434 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8566
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

20:01:08.975 | INFO    | Flow run 'literate-badger' - Beginning flow run 'literate-badger' for flow 'test-clean-chunks'

20:01:09.657 | INFO    | Task run 'clean_chunks-a50' - Attention-Is-All-You-Need_170603762v7-9ab76e8d.pdf.json: 28 → 16 chunks (12 dropped)

20:01:09.693 | INFO    | Task run 'clean_chunks-a50' - BERT_181004805v2-d4016745.pdf.json: 44 → 37 chunks (7 dropped)

20:01:09.716 | INFO    | Task run 'clean_chunks-a50' - Cleaning complete: 2 file(s) written to c:\Users\nikhi\Documents\Projects\HybridRAG-Bench\data\dev\cleaned

20:01:09.728 | INFO    | Task run 'clean_chunks-a50' - Finished in state Completed()

20:01:09.810 | INFO    | Flow run 'literate-badger' - Finished in state Completed()


Cleaned files (2):
  - c:\Users\nikhi\Documents\Projects\HybridRAG-Bench\data\dev\cleaned\Attention-Is-All-You-Need_170603762v7-9ab76e8d.pdf.json
  - c:\Users\nikhi\Documents\Projects\HybridRAG-Bench\data\dev\cleaned\BERT_181004805v2-d4016745.pdf.json


In [4]:
for cleaned_path in cleaned_paths:
    with open(cleaned_path) as f:
        chunks = json.load(f)

    # load original for comparison
    original_path = PARSED_DIR / cleaned_path.name
    with open(original_path) as f:
        original_chunks = json.load(f)

    print(f"\n{'='*60}")
    print(f"File: {cleaned_path.name}")
    print(f"Original: {len(original_chunks)} chunks  →  Cleaned: {len(chunks)} chunks  ({len(original_chunks) - len(chunks)} dropped)")
    print(f"{'='*60}")

    for i, chunk in enumerate(chunks[:3]):
        text = chunk.get("text", "")
        element_type = chunk.get("type", "unknown")

        prefix = ""
        prefix_marker = "Prefix: "
        if prefix_marker in text:
            start = text.find(prefix_marker) + len(prefix_marker)
            end = text.find("; ", start)
            if end != -1:
                prefix = text[start:end]

        original_text = text
        original_text_marker = "Original: "
        if original_text_marker in text:
            start = text.find(original_text_marker) + len(original_text_marker)
            original_text = text[start:]

        print(f"\n--- Chunk {i} ({element_type}) ---")
        if prefix:
            print(f"Prefix: {prefix}")
        print("Original: " + original_text)

    if len(chunks) > 3:
        print(f"\n... and {len(chunks) - 3} more chunk(s)")


File: Attention-Is-All-You-Need_170603762v7-9ab76e8d.pdf.json
Original: 28 chunks  →  Cleaned: 16 chunks  (12 dropped)

--- Chunk 0 (CompositeElement) ---
Prefix: This chunk is the title, author list, and abstract of the research paper "Attention Is All You Need," which introduces the Transformer model.
Original: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.

Attention Is All You Need

Ashish Vaswani∗ Google Brain avaswani@google.com

Llion Jones∗ Google Research llion@google.com

Noam Shazeer∗ Google Brain noam@google.com

Niki Parmar∗ Google Research nikip@google.com

Jakob Uszkoreit∗ Google Research usz@google.com

Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu

Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com

Illia Polosukhin∗ ‡

illia.polosukhin@gmail.com

Abstract

The dominant sequence transduction models are based on complex recurren

# Create Documents

In [5]:
from src.ingestion_steps import create_documents


@flow(name="test-create-documents", log_prints=True)
def test_create_documents(cleaned_paths: list[Path]) -> list:
    """Thin wrapper flow so the Prefect task has a flow context."""
    return create_documents(cleaned_paths=cleaned_paths)


documents = test_create_documents(cleaned_paths=cleaned_paths)
print(f"\nTotal Document(s): {len(documents)}")

20:01:17.440 | INFO    | Flow run 'proficient-tuna' - Beginning flow run 'proficient-tuna' for flow 'test-create-documents'

20:01:17.460 | INFO    | Task run 'create_documents-271' - Attention-Is-All-You-Need_170603762v7-9ab76e8d.pdf.json: created 16 Document(s)

20:01:17.472 | INFO    | Task run 'create_documents-271' - BERT_181004805v2-d4016745.pdf.json: created 37 Document(s)

20:01:17.481 | INFO    | Task run 'create_documents-271' - Total: 53 Document(s) from 2 file(s)

20:01:17.495 | INFO    | Task run 'create_documents-271' - Finished in state Completed()

20:01:17.573 | INFO    | Flow run 'proficient-tuna' - Finished in state Completed()


Total Document(s): 53


In [6]:
for i, doc in enumerate(documents[:5]):
    print(f"\n{'='*60}")
    print(f"Document {i}")
    print(f"{'='*60}")
    print(f"page_content[:200]: {doc.page_content[:200]}...")
    print(f"\nMetadata:")
    for key, value in doc.metadata.items():
        if key == "original_text":
            print(f"  {key}: {str(value)[:100]}...")
        elif key == "context_prefix":
            print(f"  {key}: {str(value)[:100]}...")
        else:
            print(f"  {key}: {value}")

if len(documents) > 5:
    print(f"\n... and {len(documents) - 5} more Document(s)")


Document 0
page_content[:200]: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.

Attention Is All You Need...

Metadata:
  context_prefix: This chunk is the title, author list, and abstract of the research paper "Attention Is All You Need,...
  paper_title: Attention Is All You Need
  section: None
  page_number: 1
  source_filename: Attention-Is-All-You-Need_170603762v7-9ab76e8d.pdf
  has_table: False
  chunk_index: 0
  element_id: c1fd5f43e33e6a757e08ca7b6effb177

Document 1
page_content[:200]: ∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started the effort to evaluate this idea. Ashish, with Illia, designed and implemented the first Tra...

Metadata:
  context_prefix: This chunk, from the 2017 NIPS paper "Attention Is All You Need," details the individual contributio...
  paper_title: Attention Is All You

# Embed

In [7]:
from src.ingestion_steps import embed


@flow(name="test-embed", log_prints=True)
def test_embed(documents: list) -> list[dict]:
    """Thin wrapper flow so the Prefect task has a flow context."""
    return embed(documents=documents)


vectors = test_embed(documents=documents)
print(f"\nTotal vector records: {len(vectors)}")

20:01:24.759 | INFO    | Flow run 'easygoing-seagull' - Beginning flow run 'easygoing-seagull' for flow 'test-embed'

20:01:24.815 | INFO    | Task run 'embed-54c' - Embedding 53 chunks (model=text-embedding-3-small, dim=1536)

20:01:29.787 | INFO    | Task run 'embed-54c' - Dense: 53 vectors, 1536-dim

20:01:41.207 | INFO    | Task run 'embed-54c' - Sparse: 53 BM25 vectors

20:01:41.212 | INFO    | Task run 'embed-54c' - Built 53 vector records for upsert

20:01:41.223 | INFO    | Task run 'embed-54c' - Finished in state Completed()

20:01:42.424 | INFO    | Flow run 'easygoing-seagull' - Finished in state Completed()


Total vector records: 53


In [8]:
dense_dim = len(vectors[0]["values"])
sparse_nnz = [len(v["sparse_values"]["indices"]) for v in vectors]
avg_sparse = sum(sparse_nnz) / len(sparse_nnz)

print(f"Dense vector dimensions: {dense_dim}")
print(f"Sparse vector non-zero entries: avg={avg_sparse:.1f}, min={min(sparse_nnz)}, max={max(sparse_nnz)}")

for i, vec in enumerate(vectors[:3]):
    print(f"\n{'='*60}")
    print(f"Vector {i}")
    print(f"{'='*60}")
    print(f"  id: {vec['id']}")
    print(f"  dense: [{vec['values'][0]:.6f}, {vec['values'][1]:.6f}, ...] ({len(vec['values'])}-dim)")
    print(f"  sparse: {len(vec['sparse_values']['indices'])} non-zero entries")
    print(f"  metadata:")
    for key, value in vec["metadata"].items():
        if key == "text":
            print(f"    {key}: {str(value)[:100]}...")
        elif key == "context_prefix":
            print(f"    {key}: {str(value)[:100]}...")
        else:
            print(f"    {key}: {value}")

if len(vectors) > 3:
    print(f"\n... and {len(vectors) - 3} more vector(s)")

Dense vector dimensions: 1536
Sparse vector non-zero entries: avg=117.2, min=55, max=191

Vector 0
  id: c1fd5f43e33e6a757e08ca7b6effb177
  dense: [0.014117, 0.027317, ...] (1536-dim)
  sparse: 133 non-zero entries
  metadata:
    text: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and...
    context_prefix: This chunk is the title, author list, and abstract of the research paper "Attention Is All You Need,...
    paper_title: Attention Is All You Need
    page_number: 1
    source_filename: Attention-Is-All-You-Need_170603762v7-9ab76e8d.pdf
    has_table: False
    chunk_index: 0

Vector 1
  id: 14984aeaed3c8ebc78f0249a40f9ebeb
  dense: [0.028478, -0.020061, ...] (1536-dim)
  sparse: 95 non-zero entries
  metadata:
    text: ∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and ...
    context_prefix: This chunk, from the 2017 NIPS paper "Attention Is All You Need," details the individual 

# Upsert to Pinecone

In [9]:
from src.ingestion_steps import upsert_to_pinecone


@flow(name="test-upsert", log_prints=True)
def test_upsert(vectors: list[dict], env: str = "dev") -> None:
    """Thin wrapper flow so the Prefect task has a flow context."""
    upsert_to_pinecone(vectors=vectors, env=env)


test_upsert(vectors=vectors, env=ENV)

20:01:57.199 | INFO    | Flow run 'ninja-sturgeon' - Beginning flow run 'ninja-sturgeon' for flow 'test-upsert'

20:01:59.535 | INFO    | Task run 'upsert_to_pinecone-e9c' - Target index: arxiv-research-rag-dev

20:02:06.142 | INFO    | Task run 'upsert_to_pinecone-e9c' - Index 'arxiv-research-rag-dev' is ready

20:02:10.751 | INFO    | Task run 'upsert_to_pinecone-e9c' - Upserted 53 vectors into 'arxiv-research-rag-dev'

20:02:10.841 | INFO    | Task run 'upsert_to_pinecone-e9c' - Index stats: 53 total vectors

20:02:10.861 | INFO    | Task run 'upsert_to_pinecone-e9c' - Finished in state Completed()

20:02:11.234 | INFO    | Flow run 'ninja-sturgeon' - Finished in state Completed()

In [10]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index("arxiv-research-rag-dev")
stats = index.describe_index_stats()

print(f"Total vectors: {stats.total_vector_count}")
print(f"Dimensions: {stats.dimension}")
print(f"Index fullness: {stats.index_fullness}")

Total vectors: 53
Dimensions: 1536
Index fullness: 0.0
